# Replicant — Cross-Source Comparison

Loads `results/results.csv` (in_process), `results/results-docker.csv`, and `results/results-k8s.csv` and shows the same scenarios side-by-side across deployment targets. The point is descriptive — *what does the framework produce under different deployments?* — not a hypothesis test.

**Configurable: which sources to include.** in_process convergence is much slower than docker/k8s on relay-heavy round_robin topologies, so its bars dominate the y-axis when all three are plotted together and the docker-vs-k8s comparison becomes hard to read. The next cell defines `INCLUDE` — default `["docker", "k8s"]`. Add `"in_process"` to the list to bring it back.

Three views:
1. **Per-scenario convergence by source** — same scenario across the loaded deployments, side-by-side.
2. **Deployment overhead ratios** — each non-baseline source as a ratio to `BASELINE` (default `docker`), to localize where the gap concentrates.
3. **Edges vs convergence at fixed N** — for each loaded source, a scatter of `edge_count` against `mean_ms` for the scaling topologies (line, ring, star, full_mesh), plus the Spearman rank correlation per (source, N, write_pattern) cell. Descriptive: what shape does the data take?

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

REPO    = Path("..").resolve()
RESULTS = REPO / "results"
FIGS    = REPO / "analysis" / "figures" / "comparison"
FIGS.mkdir(parents=True, exist_ok=True)

# All known sources. INCLUDE selects which to load. Add "in_process" to
# INCLUDE when you want the round_robin artifact visible (then probably set
# BASELINE = "in_process" too).
SOURCES = {
    "in_process": RESULTS / "results.csv",
    "docker":     RESULTS / "results-docker.csv",
    "k8s":        RESULTS / "results-k8s.csv",
}
INCLUDE  = ["docker", "k8s"]   # subset of SOURCES.keys()
BASELINE = "docker"            # used for section 2 (overhead ratios); must be in INCLUDE

assert all(s in SOURCES for s in INCLUDE), f"INCLUDE references unknown source: {INCLUDE}"
assert BASELINE in INCLUDE, f"BASELINE={BASELINE!r} must be in INCLUDE={INCLUDE}"

# Fixed colour per source so every plot below reads the same way.
SRC_PALETTE = {"in_process": "#4C72B0", "docker": "#DD8452", "k8s": "#55A467"}

## Load + tag + concat

Each CSV is tagged with its `source` column then concatenated. Sources with a missing CSV are skipped, so the notebook still works mid-sweep when only one or two deployments have been benchmarked.

In [ ]:
frames = []
for src in INCLUDE:
    path = SOURCES[src]
    if not path.exists():
        print(f"[load] {path.name} missing — skipping source={src}")
        continue
    df = pd.read_csv(path)
    df["source"] = src
    frames.append(df)

if not frames:
    raise SystemExit("no result CSVs found for INCLUDE — generate via bench-docker / bench-k8s / in-process orchestrator first")

all_df  = pd.concat(frames, ignore_index=True)
summary = all_df[all_df.row_type == "summary"].rename(columns={"trial": "n_trials"}).copy()
trials  = all_df[all_df.row_type == "trial"].copy()

summary["write_pattern"] = summary.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)
trials["write_pattern"] = trials.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)

sources_present = [s for s in INCLUDE if s in summary.source.unique()]
scenarios_per_source = {s: set(summary[summary.source == s].scenario) for s in sources_present}
shared = sorted(set.intersection(*scenarios_per_source.values())) if len(sources_present) >= 2 else []
print(f"sources loaded: {sources_present}")
print(f"scenarios per source: {{ {', '.join(f'{s}: {len(v)}' for s, v in scenarios_per_source.items())} }}")
print(f"scenarios shared across all loaded sources: {len(shared)}")

## 1. Per-scenario convergence, by source

One grouped bar per scenario × source, faceted by topology family so the y-axis can scale per facet (partition-heal and line-n10 differ by an order of magnitude). Within a facet, scenarios are ordered by node count then write pattern, so the visual scan within a topology mirrors the convergence-vs-N plot from [`convergence.ipynb`](convergence.ipynb).

Where the three bars in a triplet have the same height, deployment doesn't change the answer. Where they diverge — see the partition-heal and line round_robin scenarios — the source matters.

In [ ]:
kinds = sorted(summary.topology_kind.unique())
n_facets = len(kinds)
fig, axes = plt.subplots(n_facets, 1, figsize=(11, 3.5 * n_facets), constrained_layout=True)
if n_facets == 1:
    axes = [axes]

for ax, kind in zip(axes, kinds):
    sub = summary[summary.topology_kind == kind].copy()
    sub = sub.sort_values(["node_count", "write_pattern", "source"])
    scen_order = (
        sub.drop_duplicates("scenario")
           .sort_values(["node_count", "write_pattern"])
           .scenario.tolist()
    )
    bar_width = 0.78 / len(sources_present)
    x_base = np.arange(len(scen_order))
    for i, src in enumerate(sources_present):
        s = sub[sub.source == src].set_index("scenario").reindex(scen_order)
        offsets = x_base + (i - (len(sources_present) - 1) / 2) * bar_width
        ax.bar(
            offsets, s.mean_ms.fillna(0), width=bar_width,
            yerr=(s.p95_ms - s.mean_ms).clip(lower=0).fillna(0),
            capsize=2, color=SRC_PALETTE[src], label=src, alpha=0.9,
        )
    ax.set_xticks(x_base)
    ax.set_xticklabels(scen_order, rotation=30, ha="right")
    ax.set_ylabel("Convergence (ms)")
    ax.set_title(f"{kind}")
    ax.legend(title="source", loc="upper left", fontsize=9)

fig.suptitle("Per-scenario convergence by source (mean ± p95)", fontsize=13, y=1.005)
fig.savefig(FIGS / "by_scenario.pdf", bbox_inches="tight")
plt.show()

## 2. Deployment-overhead ratios

Per scenario, `mean_ms[source] / mean_ms[BASELINE]` for every non-baseline source. Values near 1.0 mean the deployment is at parity with the baseline; values > 1 mean slower; values < 1 mean faster.

Reading guide (default `BASELINE = "docker"`):
- A flat row of bars near 1.0 → k8s adds negligible overhead vs docker.
- A bar > 1 → real deployment overhead (kube-proxy / iptables, the extra DNS hop via the headless Service, pod scheduling, etc).
- A bar < 1 → the baseline is artifactually high for that scenario (e.g. picking `in_process` as the baseline surfaces `finding_inprocess_artifact_roundrobin` — the line/partition-heal round_robin scenarios will show ratios well below 1).

In [ ]:
if BASELINE not in sources_present:
    print(f"BASELINE={BASELINE!r} not loaded — ratio view skipped.")
else:
    others = [s for s in sources_present if s != BASELINE]
    if not others:
        print(f"only {BASELINE} loaded — nothing to compare against.")
    else:
        pivot = summary.pivot_table(
            index=["scenario", "topology_kind", "node_count"],
            columns="source",
            values="mean_ms",
        ).reset_index()
        ratios = pivot.copy()
        for s in others:
            ratios[s] = ratios[s] / ratios[BASELINE]
        ratios = ratios.dropna(subset=others, how="all").sort_values(["topology_kind", "node_count", "scenario"])

        scen_order = ratios.scenario.tolist()
        x_base = np.arange(len(scen_order))
        width = 0.78 / len(others)
        fig, ax = plt.subplots(figsize=(max(10, len(scen_order) * 0.35), 4.5))
        for i, src in enumerate(others):
            offsets = x_base + (i - (len(others) - 1) / 2) * width
            ax.bar(offsets, ratios[src], width=width,
                   color=SRC_PALETTE[src], label=f"{src} / {BASELINE}", alpha=0.9)
        ax.axhline(1.0, color="black", linestyle=":", linewidth=1)
        ax.set_xticks(x_base)
        ax.set_xticklabels(scen_order, rotation=45, ha="right")
        ax.set_ylabel(f"mean_ms / {BASELINE}.mean_ms")
        ax.set_title(f"Deployment overhead vs {BASELINE} baseline (1.0 = parity)")
        ax.legend(loc="upper left")
        fig.tight_layout()
        fig.savefig(FIGS / "overhead_ratios.pdf", bbox_inches="tight")
        plt.show()
        display_cols = list(dict.fromkeys(["scenario"] + sources_present))
        print(ratios[display_cols].round(2).to_string(index=False))

## 3. Edges vs convergence at fixed N

For each loaded source, look at how `mean_ms` relates to `edge_count` across the scaling topologies (line, ring, star, full_mesh) at a fixed N and write pattern. Two ways:

- A **Spearman rank correlation** per (source, N, write_pattern) — +1.0 = strict monotone (more edges → slower); 0 = no rank relationship; −1.0 = inverse. Spearman is rank-based, so it handles the line/star edge-count tie (both have N−1 edges) without complaint.
- A **per-cell scatter** below (one row per (N, write_pattern), one column per source). The Spearman number summarises each scatter as a single statistic.

Neither is a verdict — they're descriptive reports of what the data shows in this sweep. If you re-run with different scenarios the numbers will change; that's the point of having the framework.

In [ ]:
scaling = summary[summary.topology_kind.isin(["line", "ring", "star", "full_mesh"])].copy()

def spearman_in_group(g: pd.DataFrame) -> float:
    # Need all four topologies present for the rank to be meaningful.
    if g.topology_kind.nunique() < 4:
        return float("nan")
    # Pearson on ranks == Spearman. Avoids the scipy dependency that
    # pandas' method="spearman" lazily pulls in.
    return g["edge_count"].rank().corr(g["mean_ms"].rank())

rho = (
    scaling.groupby(["source", "node_count", "write_pattern"])
           .apply(spearman_in_group, include_groups=False)
           .rename("spearman_rho")
           .reset_index()
)
rho_wide = rho.pivot_table(
    index=["node_count", "write_pattern"],
    columns="source",
    values="spearman_rho",
)
print("Spearman ρ(edge_count, mean_ms), per (N, write_pattern) cell:\n")
print(rho_wide.round(2).fillna("—").to_string())

### Visual: convergence vs edges, per source

The scatter behind each Spearman number above. One row per (N, write_pattern), one column per loaded source. Each point is a topology; the label is the topology kind. Read it as a description of the shape, not a check against an expected ordering.

In [ ]:
import matplotlib.patches as mpatches

well_formed = (
    scaling.groupby(["source", "node_count", "write_pattern"]).topology_kind.nunique()
    == 4
)
well_formed = well_formed[well_formed].reset_index()
valid_n = sorted(set(well_formed.node_count))
patterns = ["round_robin", "concentrated"]

fig, axes = plt.subplots(
    len(valid_n) * len(patterns), len(sources_present),
    figsize=(3.4 * len(sources_present), 2.8 * len(valid_n) * len(patterns)),
    sharey=False, squeeze=False,
)
for r_idx, (n, pat) in enumerate([(n, p) for n in valid_n for p in patterns]):
    for c_idx, src in enumerate(sources_present):
        ax = axes[r_idx][c_idx]
        sub = scaling[(scaling.source == src)
                      & (scaling.node_count == n)
                      & (scaling.write_pattern == pat)]
        if sub.empty or sub.topology_kind.nunique() < 4:
            ax.text(0.5, 0.5, "—", ha="center", va="center", transform=ax.transAxes,
                    color="gray", fontsize=14)
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            sub = sub.sort_values("edge_count")
            ax.plot(sub.edge_count, sub.mean_ms, marker="o", color=SRC_PALETTE[src])
            for _, row in sub.iterrows():
                ax.annotate(row.topology_kind, (row.edge_count, row.mean_ms),
                            xytext=(4, 4), textcoords="offset points", fontsize=8,
                            color="dimgray")
            ax.set_xlabel("edges")
            ax.set_ylabel("mean_ms")
        if r_idx == 0:
            ax.set_title(src)
        if c_idx == 0:
            ax.set_ylabel(f"n={n} / {pat}\n\nmean_ms")

fig.suptitle("Convergence vs edges, per source", y=1.005)
fig.tight_layout()
fig.savefig(FIGS / "edges_vs_convergence.pdf", bbox_inches="tight")
plt.show()

## Summary table

Pivots `mean_ms` by source so the loaded deployments can be read side-by-side. When `in_process` is in `INCLUDE`, the line-n10 round_robin row surfaces the artifact described in `finding_inprocess_artifact_roundrobin`.

In [ ]:
wide = summary.pivot_table(
    index=["topology_kind", "node_count", "write_pattern", "edge_count"],
    columns="source",
    values="mean_ms",
).round(1).reset_index()
wide = wide.sort_values(["topology_kind", "node_count", "write_pattern"])
wide